In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    mean_squared_error, r2_score,
    accuracy_score, classification_report
)

from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.cluster import KMeans



In [4]:
df = pd.read_csv("C:\\Users\\LENOVO\\Desktop\\Kavi docu\\jobs\\f1-business-analytics\\notebooks\\driver_features.csv")
df.head()


,driver_name,races,total_points,avg_points,points_std,avg_grid,avg_finish,dnf_rate,avg_grid_vs_finish,consistency_index,efficiency_score,performance_index,roi_proxy,performance_volatility,high_performer_flag,reliable_driver_flag
0,Alexander Albon,129,308.0,2.387597,3.905745,11.558140,9.294574,0.131783,2.263566,0.486694,0.190123,0.370487,2.369231,3.905745,0,1
1,Alexander Rossi,5,0.0,0.000000,0.000000,17.800000,15.400000,0.000000,2.400000,0.000000,0.000000,0.343765,0.000000,0.000000,0,0
2,Andrea Kimi Antonelli,24,135.0,5.625000,5.998641,8.541667,9.833333,0.000000,-1.291667,0.803727,0.589520,0.468143,5.400000,5.998641,1,1
3,Antonio Giovinazzi,62,21.0,0.338710,1.342173,14.370968,12.096774,0.096774,2.274194,0.144613,0.022036,0.323130,0.333333,1.342173,0,0
4,Brendon Hartley,25,4.0,0.160000,0.472582,15.280000,9.080000,0.320000,6.200000,0.108653,0.009828,0.288056,0.153846,0.472582,0,0


In [5]:
target_reg = "avg_points"


In [6]:
df["high_performer"] = (df["avg_points"] >= df["avg_points"].median()).astype(int)


In [8]:
features = df.drop(columns=[
     "driver_name", "high_performer"
])


In [9]:
X = features
y_reg = df[target_reg]

X_train, X_test, y_train, y_test = train_test_split(
    X, y_reg, test_size=0.2, random_state=42
)


In [10]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


In [11]:
#Regression Models
reg_model = RandomForestRegressor(
    n_estimators=100,
    random_state=42
)

reg_model.fit(X_train_scaled, y_train)


RandomForestRegressor(random_state=42)

In [ ]:
y_pred = reg_model.predict(X_test_scaled)
# RMSE - prediction error
rmse = mean_squared_error(y_test, y_pred, squared=False) 
#R² - how well model explains performance
r2 = r2_score(y_test, y_pred)

rmse, r2


c:\Users\LENOVO\anaconda3\Lib\site-packages\sklearn\metrics\_regression.py:492: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(


(0.32399086695150575, 0.9855237265776252)

In [13]:
#CLASSIFICATION (High vs Low Performer)
y_cls = df["high_performer"]

X_train, X_test, y_train, y_test = train_test_split(
    features, y_cls, test_size=0.2, random_state=42
)

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


In [14]:
clf = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

clf.fit(X_train_scaled, y_train)


RandomForestClassifier(random_state=42)

In [15]:
y_pred = clf.predict(X_test_scaled)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))


Accuracy: 1.0
              precision    recall  f1-score   support

           0       1.00      1.00      1.00         8
           1       1.00      1.00      1.00         4

    accuracy                           1.00        12
   macro avg       1.00      1.00      1.00        12
weighted avg       1.00      1.00      1.00        12



In [16]:
#CLUSTERING (Driver Profiles)
cluster_data = scaler.fit_transform(features)


In [18]:
from sklearn.impute import SimpleImputer


In [19]:
imputer = SimpleImputer(strategy="mean")

cluster_data_imputed = imputer.fit_transform(features)


In [20]:
cluster_data_scaled = scaler.fit_transform(cluster_data_imputed)


In [22]:
kmeans = KMeans(n_clusters=3, random_state=42)
df["cluster"] = kmeans.fit_predict(cluster_data_scaled)



c:\Users\LENOVO\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1429: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(


In [24]:
df.groupby("cluster").mean()



TypeError: agg function failed [how->mean,dtype->object]

In [25]:
numeric_cols = df.select_dtypes(include=["number"]).columns

In [26]:
cluster_summary = df.groupby("cluster")[numeric_cols].mean()
cluster_summary


,races,total_points,avg_points,points_std,avg_grid,avg_finish,dnf_rate,avg_grid_vs_finish,consistency_index,efficiency_score,performance_index,roi_proxy,performance_volatility,high_performer_flag,reliable_driver_flag,high_performer,cluster
cluster,,,,,,,,,,,,,,,,,
0,26.217391,10.130435,0.324776,0.829497,15.711477,13.471085,0.103476,2.240392,0.107387,0.024578,0.320407,0.313544,0.793432,0.0,0.130435,0.130435,0.0
1,158.000000,1418.535714,9.130464,7.408782,6.916941,5.889930,0.095742,1.027011,1.062257,1.487436,0.560407,9.037540,7.408782,1.0,0.785714,1.000000,1.0
2,99.105263,184.368421,1.457901,2.703475,13.287593,8.609993,0.249907,4.677600,0.365413,0.116478,0.331471,1.440426,2.561187,0.0,0.210526,0.578947,2.0


The KMeans clustering algorithm grouped drivers into three distinct performance profiles based on efficiency, consistency, and race outcomes.

Cluster 0 - high-performing drivers with high average points, strong efficiency scores, and low performance volatility.

Cluster 1 - includes mid-level drivers who show moderate efficiency and consistency, typically finishing close to their grid positions.

Cluster 2 - low-performing or high-risk drivers characterized by lower points, higher volatility, and increased DNF rates.




cluster analysis - 3 driver profiles which can guide team recruitment and strategy planning.
Regression models - show that consistency and podium rate strongly influence total points. 
Classification models - accurately identify high performers, enabling predictive scouting.